# avgpool-reduce — worked example 2: Average pooling with a rectangular (non-square) window

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `avgpool-reduce`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

AvgPool windows do not have to be square. You can factor `H -> (h ph)` and `W -> (w pw)` with **different** factor sizes `ph` and `pw`, then mean-reduce both inner axes. This produces a `(B, C, H // ph, W // pw)` output where each entry is the mean of a `ph x pw` block.

## Worked solution

**Step 1 — state the contract.** Input `(B, C, H, W)`, window `(ph, pw)` with `H % ph == 0` and `W % pw == 0`. Output is `(B, C, H // ph, W // pw)`, each value the mean of a `ph x pw` tile.

**Step 2 — factor both spatial axes independently.** The source pattern is `b c (h p1) (w p2)`. Here `p1` is the vertical window height and `p2` is the horizontal window width — they are allowed to differ. This is the whole point of the exercise: the square-window case is just `p1 == p2`.

**Step 3 — reduce the inner factors.** Write the right side as `b c h w`, dropping both `p1` and `p2`. With `'mean'`, einops averages over each `p1 x p2` tile.

**Step 4 — supply both factor sizes.** Pass `p1=ph, p2=pw`. einops then infers `h = H // ph` and `w = W // pw`.

**Step 5 — verify.** `F.avg_pool2d` accepts a tuple `kernel_size=(ph, pw)`, which is the exact ground truth for a rectangular non-overlapping window.

In [ ]:
import torch as t
import torch.nn.functional as F
import einops
from torch import Tensor

t.manual_seed(0)

def avgpool_rect(x: Tensor, ph: int, pw: int) -> Tensor:
    return einops.reduce(
        x,
        'b c (h p1) (w p2) -> b c h w',
        'mean',
        p1=ph, p2=pw,
    )

x = t.randn(2, 4, 6, 8)
out = avgpool_rect(x, 2, 4)
ref = F.avg_pool2d(x, kernel_size=(2, 4))
print('out shape:', tuple(out.shape))
print('matches F.avg_pool2d:', bool(t.allclose(out, ref, atol=1e-6)))